# 01 — Data Understanding con módulos MLE

Este notebook introduce el cambio desde un análisis aislado hacia un proyecto modular.

**Objetivos:**
- Cargar datos mediante `src.data.load_customer_data`.
- Validar automáticamente estructura y reglas de negocio.
- Explorar calidad, distribución y señales de churn.
- Separar notebook, lógica reutilizable y configuración.


## 1. Configuración del entorno

Ejecute este notebook desde la raíz del repositorio.

En Google Colab, primero clone el repositorio y ubíquese dentro de él:

```python
!git clone <URL_DEL_REPOSITORIO>
%cd customer-intelligence-ml-platform
```


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


## 2. Importación de módulos del proyecto


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_customer_data, validate_customer_data
from src.utils import get_project_path, get_logger

logger = get_logger("notebook.data_understanding")


## 3. Carga y validación del dataset


In [ ]:
DATA_PATH = get_project_path("data", "raw", "customer_churn.csv")

df = load_customer_data(
    DATA_PATH,
    validate=True,
    require_target=True,
    minimum_rows=100,
)

logger.info("Dataset loaded successfully with shape %s", df.shape)
df.head()


In [ ]:
report = validate_customer_data(df)
report.to_dict()


## 4. Inspección general

En esta sección se revisan dimensiones, tipos de datos y primeras observaciones.


In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


## 5. Calidad de datos


In [ ]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: x["missing_count"] / len(df) * 100)
      .query("missing_count > 0")
      .sort_values("missing_pct", ascending=False)
)

missing_summary


In [ ]:
duplicate_ids = df["customer_id"].duplicated().sum()
duplicate_rows = df.duplicated().sum()

print(f"Duplicated customer IDs: {duplicate_ids}")
print(f"Duplicated complete rows: {duplicate_rows}")


## 6. Distribución de la variable objetivo


In [ ]:
target_summary = (
    df["churn"]
    .value_counts(dropna=False)
    .rename_axis("churn")
    .to_frame("count")
)

target_summary["percentage"] = target_summary["count"] / len(df) * 100
target_summary


In [ ]:
ax = df["churn"].value_counts().sort_index().plot(kind="bar")
ax.set_title("Customer churn distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Customers")
plt.show()


## 7. Análisis numérico


In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
numeric_columns


In [ ]:
correlations = (
    df[numeric_columns]
    .corr(numeric_only=True)["churn"]
    .drop("churn")
    .sort_values(key=abs, ascending=False)
)

correlations.to_frame("correlation_with_churn")


In [ ]:
selected_numeric = [
    "tenure_months",
    "monthly_fee",
    "support_calls",
    "complaints",
    "last_payment_delay",
    "digital_usage_score",
]

df[selected_numeric].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()


## 8. Análisis categórico


In [ ]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()
categorical_columns


In [ ]:
def churn_rate_by_category(dataframe: pd.DataFrame, column: str) -> pd.DataFrame:
    return (
        dataframe.groupby(column, dropna=False)["churn"]
        .agg(customer_count="size", churn_rate="mean")
        .sort_values("churn_rate", ascending=False)
    )

churn_rate_by_category(df, "contract_type")


In [ ]:
churn_rate_by_category(df, "internet_service")


In [ ]:
churn_rate_by_category(df, "payment_method")


## 9. Hallazgos preliminares

Complete esta sección con conclusiones sustentadas en resultados:

1. ¿Qué variables presentan valores faltantes?
2. ¿Existe desbalance en `churn`?
3. ¿Qué variables numéricas tienen mayor relación con el target?
4. ¿Qué categorías muestran mayor tasa de churn?
5. ¿Qué variables deben excluirse antes del modelado?


In [ ]:
student_findings = {
    "missing_values": "",
    "class_balance": "",
    "numeric_signals": "",
    "categorical_signals": "",
    "variables_to_exclude": "",
}

student_findings


## 10. Checklist de salida

Antes de cerrar el notebook, verifique:

- [ ] El dataset fue cargado usando `load_customer_data`.
- [ ] La validación terminó sin errores.
- [ ] Se revisaron valores faltantes y duplicados.
- [ ] Se analizó la distribución del target.
- [ ] Se identificaron variables candidatas para el modelo.
- [ ] Se documentaron decisiones preliminares de limpieza.


## Resultado esperado

Este notebook ya no contiene lógica crítica de carga o validación.  
La lógica reusable reside en `src/`, mientras que el notebook conserva la exploración, interpretación y comunicación de resultados.
